# Chapter 8: Memory Management

EliteAI Academy

Manage conversation windows, sessions, long-term stores, semantic/episodic/procedural memory, and cross-session recall.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os, json, re, time, uuid
from typing import List, Dict, Optional, Any
from dataclasses import dataclass, field
from datetime import datetime

PROVIDER = "deepseek"

if PROVIDER == "deepseek":
    import anthropic
    _client = anthropic.Anthropic(timeout=60.0, max_retries=2)
    _model  = MODEL

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.",
                 history: List[Dict] = None, max_tokens: int = 512) -> str:
        messages = list(history or []) + [{"role": "user", "content": prompt}]
        r = _client.messages.create(thinking={"type": "disabled"}, 
            model=_model, max_tokens=max_tokens, system=system, messages=messages
        )
        return r.content[0].text.strip()

elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    _gmodel = genai.GenerativeModel("gemini-2.0-flash")

    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.",
                 history: List[Dict] = None, max_tokens: int = 512) -> str:
        ctx = ""
        if history:
            ctx = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in history) + "\n\n"
        return _gmodel.generate_content(f"{system}\n\n{ctx}USER: {prompt}").text.strip()

print(f"✅ Provider: {PROVIDER}")
print(call_llm("Reply with exactly: MEMORY READY"))

## 1. The Problem — Stateless Agents Have No Memory

Without memory, every message is a fresh start. The agent cannot recall anything said earlier in the conversation.

In [ ]:
print("=== STATELESS AGENT (no memory) ===")
print()

turns = [
    "Hi! My name is Phunkie and I'm a data engineer.",
    "I'm working on a project using Azure Synapse Analytics.",
    "What was my name again, and what project am I working on?",
]

for turn in turns:

    response = call_llm(turn, system="You are a helpful assistant.")
    print(f"👤 User: {turn}")
    print(f"🤖 Agent: {response}")
    print()

print("⚠️  The agent cannot recall the user's name or project.")
print("   Each call is completely independent — total amnesia.")

## 2. Memory Architecture Overview

In [ ]:
architecture = """
╔══════════════════════════════════════════════════════════════════════╗
║                    AGENT MEMORY ARCHITECTURE                         ║
╠══════════════════════════════════════════╦═══════════════════════════╣
║  SHORT-TERM MEMORY                       ║  LONG-TERM MEMORY         ║
║  (Contextual / Working)                  ║  (Persistent)             ║
╠══════════════════════════════════════════╬═══════════════════════════╣
║  • Lives in the LLM context window       ║  • External database      ║
║  • Current conversation history          ║  • Vector store           ║
║  • Tool call results                     ║  • Key-value store        ║
║  • Agent scratch pad / reflections       ║  • Knowledge graph        ║
║  • Lost when session ends                ║  • Persists across runs   ║
║  • Limited by token window               ║  • Unlimited (with cost)  ║
╠══════════════════════════════════════════╬═══════════════════════════╣
║  TYPES OF LONG-TERM MEMORY                                           ║
║  • Semantic   → Facts & preferences ("user likes Python")            ║
║  • Episodic   → Past experiences ("last week we solved X bug")       ║
║  • Procedural → How-to rules ("always format output as JSON")        ║
╠══════════════════════════════════════════╬═══════════════════════════╣
║  SESSION / STATE                         ║  MEMORY SERVICE           ║
║  (ADK concepts)                          ║  (ADK concept)            ║
║  • Current thread context                ║  • Cross-session search   ║
║  • Key-value state dict                  ║  • Semantic retrieval     ║
║  • Scoped: session / user / app          ║  • add + search methods   ║
╚══════════════════════════════════════════╩═══════════════════════════╝
"""
print(architecture)

## 3. Short-Term Memory — Conversation History

The simplest form: pass the full conversation history to every LLM call.

In [ ]:
class ConversationHistory:
    'Manages short-term conversation memory.'
    def __init__(self, system: str = "You are a helpful assistant."):
        self.system  = system
        self.history: List[Dict] = []
        self.turn_count = 0

    def chat(self, user_message: str) -> str:
        'Send a message and get a response, maintaining full history.'

        response = call_llm(user_message, system=self.system, history=self.history)

        self.history.append({"role": "user",      "content": user_message})
        self.history.append({"role": "assistant", "content": response})
        self.turn_count += 1
        return response

    def token_estimate(self) -> int:
        'Rough token estimate (4 chars ≈ 1 token).'
        total_chars = sum(len(m["content"]) for m in self.history)
        return total_chars // 4

    def display(self):
        print(f"\n📜 History ({self.turn_count} turns, ~{self.token_estimate()} tokens):")
        for m in self.history:
            icon = "👤" if m["role"] == "user" else "🤖"
            print(f"  {icon} {m['role'].upper()}: {m['content'][:100]}")

print("=== WITH SHORT-TERM MEMORY ===")
conv = ConversationHistory(system="You are a helpful technical assistant. Remember everything the user tells you.")

turns = [
    "Hi! My name is Phunkie and I'm a data engineer.",
    "I'm working on a project using Azure Synapse Analytics.",
    "The project involves migrating from on-premise SQL Server.",
    "What was my name again, and summarise what you know about my project?",
]

for turn in turns:
    response = conv.chat(turn)
    print(f"\n👤 {turn}")
    print(f"🤖 {response}")

conv.display()

## 4. Context Window Management — Sliding Window

As conversations grow, they exceed the context window. A sliding window keeps only the most recent N turns.

In [ ]:
class SlidingWindowMemory:
    'Keeps only the last `max_turns` conversation turns.'
    def __init__(self, system: str, max_turns: int = 5):
        self.system    = system
        self.max_turns = max_turns
        self.history:  List[Dict] = []
        self.all_turns: int = 0

    def _windowed_history(self) -> List[Dict]:
        'Return only the last max_turns pairs (each pair = user + assistant).'
        max_messages = self.max_turns * 2
        return self.history[-max_messages:] if len(self.history) > max_messages else self.history

    def chat(self, user_message: str) -> str:
        windowed = self._windowed_history()
        response = call_llm(user_message, system=self.system, history=windowed)
        self.history.append({"role": "user",      "content": user_message})
        self.history.append({"role": "assistant", "content": response})
        self.all_turns += 1
        dropped = max(0, len(self.history) // 2 - self.max_turns)
        print(f"  📊 Turn {self.all_turns}: total={len(self.history)//2} turns | "
              f"in-window={len(windowed)//2} | dropped={dropped}")
        return response

class SummarisedMemory:
    'When history exceeds threshold, summarise old turns into a compact summary.'
    def __init__(self, system: str, max_turns: int = 4):
        self.system    = system
        self.max_turns = max_turns
        self.history:  List[Dict] = []
        self.summary:  str = ""

    def _maybe_summarise(self):
        'If history is too long, compress old turns into a summary.'
        if len(self.history) > self.max_turns * 2:

            to_compress  = self.history[:self.max_turns]
            self.history = self.history[self.max_turns:]

            text = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in to_compress)
            new_summary = call_llm(
                f"Summarise this conversation in 3 sentences, preserving key facts:\n\n{text}"
            )
            self.summary = f"{self.summary} {new_summary}".strip() if self.summary else new_summary
            print(f"  🗜️  Compressed {len(to_compress)//2} turns into summary")

    def chat(self, user_message: str) -> str:
        context_system = self.system
        if self.summary:
            context_system += f"\n\nConversation summary so far:\n{self.summary}"

        response = call_llm(user_message, system=context_system, history=self.history)
        self.history.append({"role": "user",      "content": user_message})
        self.history.append({"role": "assistant", "content": response})
        self._maybe_summarise()
        return response

long_convo = [
    "My name is Phunkie.",
    "I work at Wragby Solutions.",
    "My current project is the Seplat EDW.",
    "I use Azure Synapse and Databricks.",
    "My favourite programming language is Python.",
    "I also know C# and SQL very well.",
    "What do you know about me so far?",
]

print("=== SLIDING WINDOW (max 3 turns) ===")
sw = SlidingWindowMemory(system="You are a helpful assistant. Remember user details.", max_turns=3)
for t in long_convo:
    r = sw.chat(t)
    if t == long_convo[-1]:
        print(f"\n🤖 {r}")

print("\n" + "="*50)
print("=== SUMMARISED MEMORY (max 4 turns then compress) ===")
sm = SummarisedMemory(system="You are a helpful assistant.", max_turns=4)
for t in long_convo:
    r = sm.chat(t)
    if t == long_convo[-1]:
        print(f"\n🤖 {r}")
        if sm.summary:
            print(f"\n📝 Stored Summary:\n{sm.summary}")

## 5. Session & State Management

A `Session` is a named conversation thread. `State` is a key-value scratchpad that persists within (and optionally beyond) the session.

In [ ]:
@dataclass
class Session:
    'Represents a single conversation thread.'
    session_id: str
    user_id:    str
    app_name:   str
    messages:   List[Dict] = field(default_factory=list)
    state:      Dict[str, Any] = field(default_factory=dict)
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    updated_at: str = field(default_factory=lambda: datetime.now().isoformat())

    def add_message(self, role: str, content: str):
        self.messages.append({"role": role, "content": content})
        self.updated_at = datetime.now().isoformat()

    def update_state(self, updates: Dict[str, Any]):
        'Update state using key prefixes to control scope.'
        self.state.update(updates)
        self.updated_at = datetime.now().isoformat()

    def get_state(self, key: str, default=None):
        return self.state.get(key, default)

    def display(self):
        print(f"\n📋 Session: {self.session_id}")
        print(f"   User: {self.user_id} | App: {self.app_name}")
        print(f"   Messages: {len(self.messages)} | Last updated: {self.updated_at[:19]}")
        print(f"   State: {json.dumps(self.state, indent=4)}")

class InMemorySessionService:
    'Simple in-memory session store (mirrors ADK InMemorySessionService).'

    def __init__(self):
        self._sessions: Dict[str, Session] = {}

    def create_session(self, app_name: str, user_id: str,
                       session_id: str = None, state: Dict = None) -> Session:
        sid = session_id or str(uuid.uuid4())[:8]
        session = Session(session_id=sid, user_id=user_id,
                         app_name=app_name, state=state or {})
        self._sessions[sid] = session
        print(f"✅ Session created: {sid} (user={user_id}, app={app_name})")
        return session

    def get_session(self, session_id: str) -> Optional[Session]:
        return self._sessions.get(session_id)

    def delete_session(self, session_id: str):
        self._sessions.pop(session_id, None)
        print(f"🗑️  Session deleted: {session_id}")

    def list_sessions(self, user_id: str) -> List[Session]:
        return [s for s in self._sessions.values() if s.user_id == user_id]

service = InMemorySessionService()
session = service.create_session(
    app_name="support_bot",
    user_id="phunkie_001",
    state={"task_status": "idle", "user:preferred_language": "Python"}
)

print(f"\nInitial state: {session.state}")

session.add_message("user", "I need help with my Azure pipeline.")
session.update_state({"task_status": "active", "user:current_topic": "Azure pipeline", "turn_count": 1})

session.add_message("assistant", "I can help! What specific issue are you facing?")
session.add_message("user", "The ADF pipeline is failing on the copy activity.")
session.update_state({"task_status": "diagnosing", "turn_count": 2, "temp:last_issue": "ADF copy activity failure"})

session.display()

## 6. Long-Term Memory — Semantic Memory Store

Stores facts about users/entities that persist across sessions. Retrieves them via keyword or semantic search.

In [ ]:
@dataclass
class MemoryEntry:
    'A single long-term memory record.'
    key:        str
    content:    str
    namespace:  str
    metadata:   Dict = field(default_factory=dict)
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    entry_id:   str = field(default_factory=lambda: str(uuid.uuid4())[:8])

class InMemoryLongTermStore:
    'Simple long-term memory store with namespace support.'
    def __init__(self):
        self._store: Dict[str, Dict[str, MemoryEntry]] = {}

    def put(self, namespace: str, key: str, content: str, metadata: Dict = None):
        'Store a memory under namespace:key.'
        if namespace not in self._store:
            self._store[namespace] = {}
        self._store[namespace][key] = MemoryEntry(
            key=key, content=content, namespace=namespace, metadata=metadata or {}
        )
        print(f"  💾 Stored [{namespace}/{key}]: {content[:60]}")

    def get(self, namespace: str, key: str) -> Optional[MemoryEntry]:
        'Retrieve a specific memory by namespace and key.'
        return self._store.get(namespace, {}).get(key)

    def search(self, namespace: str, query: str, top_k: int = 3) -> List[MemoryEntry]:
        'Semantic-style search within a namespace.'
        entries = list(self._store.get(namespace, {}).values())
        if not entries:
            return []

        entries_text = "\n".join(
            f"{i}. [{e.key}]: {e.content}" for i, e in enumerate(entries)
        )
        raw = call_llm(
            f"""Given this query: '{query}'
Rate each memory's relevance (0-10) and return a JSON array of [index, score] pairs.
Return ONLY a JSON array, e.g. [[0, 8], [1, 3], [2, 6]]

Memories:\n{entries_text}""",
            system="You are a relevance scorer. Return only valid JSON."
        )
        try:
            scores = json.loads(re.sub(r"```[\w]*", "", raw).strip())
            ranked = sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]
            return [entries[idx] for idx, _ in ranked if idx < len(entries)]
        except:
            return entries[:top_k]

    def list_namespaces(self) -> List[str]:
        return list(self._store.keys())

    def summary(self):
        total = sum(len(v) for v in self._store.values())
        print(f"\n📚 Memory Store: {len(self._store)} namespaces | {total} total entries")
        for ns, entries in self._store.items():
            print(f"   [{ns}]: {len(entries)} entries")

store = InMemoryLongTermStore()

print("=== STORING LONG-TERM MEMORIES ===")
store.put("phunkie/profile",      "name",     "User's name is Phunkie (Funke Yusuf)")
store.put("phunkie/profile",      "role",     "Data engineer and AI/ML practitioner")
store.put("phunkie/profile",      "company",  "Works at Wragby Solutions on Seplat Energy EDW project")
store.put("phunkie/preferences",  "language", "Preferred language: Python, also knows C# and SQL")
store.put("phunkie/preferences",  "tools",    "Uses Azure Synapse, Databricks, ADF, Power BI")
store.put("phunkie/history",      "proj_001", "Completed PhishMe security pipeline with PySpark ingestion")
store.put("phunkie/history",      "proj_002", "Working on SAP ECC to Azure lakehouse migration")

print("\n=== SEARCHING MEMORIES ===")
results = store.search("phunkie/profile", "what does this user do for work?", top_k=2)
print("\nSearch results for 'what does this user do for work?':")
for r in results:
    print(f"  [{r.key}]: {r.content}")

results2 = store.search("phunkie/history", "Azure and data engineering projects", top_k=2)
print("\nSearch results for 'Azure and data engineering projects':")
for r in results2:
    print(f"  [{r.key}]: {r.content}")

store.summary()

## 7. Memory-Augmented Agent

Combining short-term (conversation history) + long-term (persistent store) into a single coherent agent that remembers across sessions.

In [ ]:
class MemoryAugmentedAgent:
    'Full memory-augmented agent:'
    def __init__(self, name: str, user_id: str,
                 long_term_store: InMemoryLongTermStore,
                 session_service: InMemorySessionService,
                 window_size: int = 6):
        self.name            = name
        self.user_id         = user_id
        self.store           = long_term_store
        self.session_service = session_service
        self.window_size     = window_size
        self.namespace       = f"{user_id}/facts"

        self.session = session_service.create_session(
            app_name=name, user_id=user_id
        )

    def _retrieve_relevant_memories(self, query: str) -> str:
        'Search long-term store for memories relevant to current query.'
        results = self.store.search(self.namespace, query, top_k=3)
        if not results:
            return ""
        return "\n".join(f"- {r.content}" for r in results)

    def _extract_and_store_facts(self, user_message: str, response: str):
        'Extract new facts from conversation and store in long-term memory.'
        facts_raw = call_llm(
            f"""Extract any NEW factual information about the user from this exchange.
Return ONLY a JSON array of strings. Each string is one fact.
If no new facts, return an empty array [].

User: {user_message}
Assistant: {response}""",
            system="You are a fact extractor. Return only valid JSON."
        )
        try:
            facts = json.loads(re.sub(r"```[\w]*", "", facts_raw).strip())
            for fact in facts:
                key = f"fact_{len(self.store._store.get(self.namespace, {})):03d}"
                self.store.put(self.namespace, key, fact)
        except:
            pass

    def chat(self, user_message: str, verbose: bool = True) -> str:
        'Process a user message with full memory support.'

        memories = self._retrieve_relevant_memories(user_message)

        system = f"""You are {self.name}, a helpful personal AI assistant.
You have a persistent memory of this user's facts and preferences.
"""
        if memories:
            system += f"\nRelevant memories about this user:\n{memories}\n"

        messages = self.session.messages
        windowed = messages[-(self.window_size * 2):] if len(messages) > self.window_size * 2 else messages

        response = call_llm(user_message, system=system, history=windowed)

        self.session.add_message("user",      user_message)
        self.session.add_message("assistant", response)

        self._extract_and_store_facts(user_message, response)

        if verbose:
            memories_count = len(self.store._store.get(self.namespace, {}))
            print(f"  [Memory: {memories_count} facts stored | {len(windowed)//2} turns in window]")

        return response

lt_store  = InMemoryLongTermStore()
ss        = InMemorySessionService()
agent     = MemoryAugmentedAgent("Aria", "phunkie_001", lt_store, ss)

print("\n=== MEMORY-AUGMENTED AGENT CONVERSATION ===")
convo = [
    "Hi! My name is Phunkie. I work as a data engineer at Wragby Solutions.",
    "I'm currently building an enterprise data warehouse using Azure Synapse.",
    "My favourite tool for data transformation is dbt.",
    "Can you remember what I told you about my role and current project?",
]

for turn in convo:
    response = agent.chat(turn)
    print(f"\n👤 {turn}")
    print(f"🤖 {response}")

print("\n── Long-term facts stored ──")
for key, entry in lt_store._store.get("phunkie_001/facts", {}).items():
    print(f"  [{key}]: {entry.content}")

## 8. LangChain — `ChatMessageHistory` and `ConversationBufferMemory`

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory as ChatMessageHistory
from langchain_classic.memory import ConversationBufferMemory
from langchain_anthropic import ChatAnthropic
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate

lc_llm = ChatAnthropic(model_kwargs={"thinking": {"type": "disabled"}}, model=MODEL, temperature=0)

print("=== A: ChatMessageHistory (Manual) ===")
history = ChatMessageHistory()
history.add_user_message("My name is Phunkie and I live in Lagos.")
history.add_ai_message("Nice to meet you, Phunkie! Lagos is a vibrant city.")
history.add_user_message("I prefer Python for data engineering work.")
history.add_ai_message("Python is excellent for data engineering — great choice!")

print(f"Stored {len(history.messages)} messages:")
for msg in history.messages:
    icon = "👤" if msg.type == "human" else "🤖"
    print(f"  {icon} {msg.type}: {msg.content}")

print("\n=== B: ConversationBufferMemory (Automatic) ===")

template = """You are a helpful technical assistant.

Previous conversation:
{history}

User: {question}
Assistant:"""

prompt   = PromptTemplate.from_template(template)
memory   = ConversationBufferMemory(memory_key="history")
chain    = LLMChain(llm=lc_llm, prompt=prompt, memory=memory)

questions = [
    "Hi, I'm Phunkie. I'm learning about agentic AI.",
    "What is the most important pattern for multi-step tasks?",
    "Based on what I told you about myself, what topic would you recommend I start with?",
]

for q in questions:
    response = chain.predict(question=q)
    print(f"\n👤 {q}")
    print(f"🤖 {response[:200]}")

print(f"\n📜 Memory buffer has {len(memory.chat_memory.messages)} messages stored")

## 9. Three Types of Long-Term Memory

**Semantic** = facts | **Episodic** = past experiences | **Procedural** = how-to rules

In [ ]:
print("=== SEMANTIC MEMORY (Facts & Preferences) ===")

user_profile = {
    "name":     "Phunkie",
    "role":     "Data Engineer",
    "language": "Python",
    "tools":    ["Azure Synapse", "Databricks", "dbt"],
    "prefers":  "short, direct explanations with code examples"
}

def semantic_memory_response(question: str, profile: dict) -> str:
    profile_text = json.dumps(profile, indent=2)
    return call_llm(
        question,
        system=f"""You are a personalised AI assistant.
Use this user profile to personalise your response:
{profile_text}"""
    )

q = "How should I approach learning multi-agent systems?"
print(f"\n❓ {q}")
print(f"🤖 {semantic_memory_response(q, user_profile)}")

print("\n=== EPISODIC MEMORY (Past Experiences) ===")

past_episodes = [
    {"task": "Fix ADF pipeline column mismatch",
     "solution": "Used schema mapping in the copy activity sink settings",
     "outcome": "Resolved — pipeline ran successfully"},
    {"task": "Handle MERGE incompatibility on REPLICATE tables",
     "solution": "Switched to truncate-staging + upsert pattern",
     "outcome": "Resolved — deployed to production"},
]

episodes_text = "\n".join(
    f"Task: {e['task']}\nSolution: {e['solution']}\nOutcome: {e['outcome']}\n"
    for e in past_episodes
)

new_problem = "My Synapse pipeline fails with a DUPLICATE_KEY error on the staging table."
print(f"\n❓ New problem: {new_problem}")
response = call_llm(
    f"New problem: {new_problem}",
    system=f"""You are a senior data engineer.
Here are past experiences solving similar problems:

{episodes_text}
Use these as reference when suggesting a solution."""
)
print(f"🤖 {response}")

print("\n=== PROCEDURAL MEMORY (Self-Updating Instructions) ===")

current_instructions = """You are a helpful assistant.
- Answer questions clearly and concisely.
- Use plain English."""

recent_feedback = [
    "User said: 'Your responses are too vague, I need code examples'",
    "User said: 'Please structure your answers with numbered steps'",
    "User said: 'I prefer Python examples over pseudocode'",
]

feedback_text = "\n".join(recent_feedback)
updated_instructions = call_llm(
    f"""Here are your current instructions:
{current_instructions}

Recent user feedback:
{feedback_text}

Rewrite the instructions to incorporate this feedback.
Return ONLY the improved instructions.""",
    system="You are a self-improving AI. Update your own instructions based on feedback."
)

print(f"\nOriginal instructions:\n{current_instructions}")
print(f"\nUpdated instructions (after reflection):\n{updated_instructions}")
current_instructions = updated_instructions

## 10. LangGraph `InMemoryStore` — Namespace-Based Memory

In [ ]:
class LangGraphStyleStore:
    'Simulates LangGraph InMemoryStore with put/get/search.'
    def __init__(self):
        self._data: Dict = {}

    def put(self, namespace: tuple, key: str, value: dict):
        ns_str = "/".join(namespace)
        if ns_str not in self._data:
            self._data[ns_str] = {}
        self._data[ns_str][key] = {"value": value, "created_at": datetime.now().isoformat()}
        print(f"  📥 store.put({namespace}, '{key}')")

    def get(self, namespace: tuple, key: str) -> Optional[dict]:
        ns_str = "/".join(namespace)
        return self._data.get(ns_str, {}).get(key)

    def search(self, namespace: tuple, query: str = None, filter_dict: dict = None) -> List[dict]:
        ns_str = "/".join(namespace)
        items  = list(self._data.get(ns_str, {}).values())
        if filter_dict:
            items = [i for i in items
                     if all(i["value"].get(k) == v for k, v in filter_dict.items())]
        return items

store = LangGraphStyleStore()

user_id = "phunkie"
app_ctx = "technical_assistant"

store.put(
    (user_id, "preferences"),
    "communication_style",
    {"rules": [
        "User prefers Python code examples",
        "User likes numbered step-by-step explanations",
        "User is an expert in Azure and Databricks"
    ]}
)

store.put(
    (user_id, "agent_instructions"),
    "system_prompt",
    {"instructions": "You are a helpful data engineering assistant specialising in Azure."}
)

store.put(
    (user_id, "episodes"),
    "ep_001",
    {"summary": "Helped user fix ADF pipeline column mismatch", "resolved": True}
)
store.put(
    (user_id, "episodes"),
    "ep_002",
    {"summary": "Explained Databricks Delta Lake merge patterns", "resolved": True}
)

prefs = store.get((user_id, "preferences"), "communication_style")
instructions = store.get((user_id, "agent_instructions"), "system_prompt")
episodes = store.search((user_id, "episodes"))

print(f"\n📋 Retrieved preferences: {prefs['value']['rules']}")
print(f"📋 Retrieved instructions: {instructions['value']['instructions']}")
print(f"📋 Retrieved {len(episodes)} episodes")

pref_text = "\n".join(f"- {r}" for r in prefs["value"]["rules"])
ep_text   = "\n".join(f"- {e['value']['summary']}" for e in episodes)
system    = f"""{instructions['value']['instructions']}

User preferences:\n{pref_text}

Past interactions:\n{ep_text}"""

response = call_llm("How do I implement slowly changing dimensions in Synapse?", system=system)
print(f"\n🤖 Personalised response:\n{response}")

## 11. Cross-Session Memory — Remembering Between Conversations

In [ ]:
class CrossSessionAgent:
    'Agent that retains long-term memories BETWEEN sessions.'
    def __init__(self, user_id: str, store: InMemoryLongTermStore):
        self.user_id    = user_id
        self.store      = store
        self.namespace  = f"{user_id}/cross_session"
        self.history    = []
        self.session_id = None

    def start_session(self) -> str:
        self.session_id = str(uuid.uuid4())[:8]
        self.history    = []
        print(f"\n🟢 New session started: {self.session_id}")

        all_memories = list(self.store._store.get(self.namespace, {}).values())
        if all_memories:
            memory_text = "\n".join(f"- {m.content}" for m in all_memories[:5])
            print(f"📚 Loaded {len(all_memories)} memories from previous sessions")
            self._session_context = f"Long-term memories from previous sessions:\n{memory_text}"
        else:
            self._session_context = ""

        return self.session_id

    def end_session(self):
        'Extract key facts from this session and store long-term.'
        if not self.history:
            return
        conv_text = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in self.history)
        facts_raw = call_llm(
            f"Extract 3 key facts from this session that should be remembered long-term:\n{conv_text}\nReturn a JSON array.",
            system="You are a memory extractor. Return only valid JSON."
        )
        try:
            facts = json.loads(re.sub(r"```[\w]*", "", facts_raw).strip())
            for i, fact in enumerate(facts):
                self.store.put(self.namespace, f"{self.session_id}_{i:02d}", fact)
            print(f"\n🔴 Session {self.session_id} ended — {len(facts)} facts stored to long-term memory")
        except:
            print(f"\n🔴 Session {self.session_id} ended")

    def chat(self, message: str) -> str:
        system = "You are a helpful technical assistant with persistent memory."
        if self._session_context:
            system += f"\n\n{self._session_context}"

        response = call_llm(message, system=system, history=self.history)
        self.history.append({"role": "user",      "content": message})
        self.history.append({"role": "assistant", "content": response})
        return response

shared_store = InMemoryLongTermStore()
agent = CrossSessionAgent("phunkie", shared_store)

print("=" * 55 + "\n SESSION 1\n" + "=" * 55)
agent.start_session()

for msg in [
    "Hi! I'm Phunkie. I'm a data engineer at Wragby Solutions.",
    "Today I solved a tricky Delta Lake schema evolution issue in Databricks.",
    "The fix was to use mergeSchema=True in the write options.",
]:
    r = agent.chat(msg)
    print(f"👤 {msg}\n🤖 {r[:150]}\n")

agent.end_session()

print("\n" + "=" * 55 + "\n SESSION 2 (new session)\n" + "=" * 55)
agent.start_session()

for msg in [
    "Hello again!",
    "What do you remember about me and what we discussed last time?",
]:
    r = agent.chat(msg)
    print(f"👤 {msg}\n🤖 {r}\n")

## 12. Production Memory Patterns — Decision Guide

In [ ]:
guide = """
╔═══════════════════════════╦══════════════════╦══════════════════════════════════╗
║ Memory Type               ║ Implementation   ║ When to Use                      ║
╠═══════════════════════════╬══════════════════╬══════════════════════════════════╣
║ Full conversation history ║ List in memory   ║ Short conversations (<20 turns)  ║
╠═══════════════════════════╬══════════════════╬══════════════════════════════════╣
║ Sliding window            ║ List[-N*2:]      ║ Long chats, cost-sensitive apps  ║
╠═══════════════════════════╬══════════════════╬══════════════════════════════════╣
║ Summarised memory         ║ LLM compression  ║ Very long sessions, strict token ║
║                           ║                  ║ budgets                          ║
╠═══════════════════════════╬══════════════════╬══════════════════════════════════╣
║ In-memory store           ║ Dict / list      ║ Development, testing, demos      ║
╠═══════════════════════════╬══════════════════╬══════════════════════════════════╣
║ SQLite / PostgreSQL        ║ SQL + ORM        ║ Small to medium production apps  ║
╠═══════════════════════════╬══════════════════╬══════════════════════════════════╣
║ Vector database           ║ Pinecone,        ║ Semantic search, RAG, large      ║
║ (semantic search)         ║ Weaviate, pgvec  ║ knowledge bases                  ║
╠═══════════════════════════╬══════════════════╬══════════════════════════════════╣
║ Managed memory service    ║ Vertex Memory    ║ Enterprise scale, GCP production  ║
║                           ║ Bank, Mem0       ║                                  ║
╚═══════════════════════════╩══════════════════╩══════════════════════════════════╝

KEY RULES:
  • Short-term: always include — no stateless agents for multi-turn conversations
  • Long-term: use for user preferences, facts, and cross-session continuity
  • State prefixes (ADK): session (none) | user: | app: | temp:
  • Never directly mutate state dict — always use append_event or output_key
  • Always cap window size or summarise to avoid context overflow
"""
print(guide)

## 13. Exercises

- **Exercise 1 — Token Budget:** Modify `SlidingWindowMemory` to track actual token usage per turn. Drop the oldest turn when the cumulative token count exceeds 2,000 (not just a fixed window).

- **Exercise 2 — Importance-Weighted Memory:** Extend `InMemoryLongTermStore` with an `importance` score (1-10) on each entry. When the store exceeds 20 entries, auto-prune the lowest-importance ones.

- **Exercise 3 — Contradiction Detection:** Before storing a new fact, check existing memories for contradictions (e.g. "user lives in Lagos" vs new "user lives in Abuja"). Resolve by keeping the newest.

- **Exercise 4 — Multi-User Memory:** Extend `CrossSessionAgent` to support multiple users. Each user has their own namespace. Build a router that loads the correct user's memories based on `user_id`.

- **Exercise 5 — Real Embeddings:** Replace the LLM-based relevance scoring in `InMemoryLongTermStore.search` with real TF-IDF cosine similarity using `sklearn.feature_extraction.text.TfidfVectorizer`.

**Next:** Chapter 9 — Learning and Adaptation